# OptiSAR Rescue — Flood Extent Mapping Prototype

Automated flood-mapping pipeline using **Google Earth Engine**, **Sentinel-1 SAR** & **Sentinel-2 optical** imagery over the **Kaziranga National Park** AOI.

**Tech Stack:** Python · Google Earth Engine API · Sentinel-1 (SAR) · Sentinel-2 (Optical) · Folium

## 1. Setup & Authentication

In [ ]:
import ee
import folium
import numpy as np

ee.Authenticate()
ee.Initialize(project='ee-optisar-test')
print("Earth Engine initialized successfully.")

## 2. Define Area of Interest (AOI)

> **Kaziranga National Park, Assam** — one of India's most flood-prone UNESCO World Heritage Sites.

In [ ]:
# Kaziranga National Park, Assam
aoi = ee.Geometry.Point([93.17, 26.58])
region = aoi.buffer(10000)  # 10 km buffer around the point

## 3. Acquire Satellite Imagery

> We use two complementary data sources:
> - **Sentinel-2 (Optical):** Cloud-masked surface reflectance for water body identification via NDWI
> - **Sentinel-1 (SAR):** C-band radar backscatter that penetrates cloud cover — critical during monsoon season

In [ ]:
# Sentinel-2: Optical imagery (least cloudy scene)
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2024-06-01', '2024-08-30')
      .sort('CLOUDY_PIXEL_PERCENTAGE')
      .first())

# NDWI: Normalized Difference Water Index (Green - NIR) / (Green + NIR)
ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')

# Sentinel-1: SAR imagery (VV polarization, Interferometric Wide swath)
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(aoi)
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filterDate('2024-06-01', '2024-08-30')
      .first()
      .select('VV'))

print("Sentinel-1 and Sentinel-2 imagery loaded successfully.")

## 4. Flood Detection — Thresholding & Segmentation

> **Optical mask (NDWI > 0):** Identifies permanent water bodies  
> **SAR mask (VV < −15 dB):** Detects all water surfaces including flood water  
> **Inundation extent = SAR water − Optical water:** Isolates *new* flooding from permanent water

In [ ]:
# Optical water mask: NDWI > 0 (standard threshold for water)
optical_water = ndwi.gt(0).rename('Optical_Water')

# SAR water mask: VV backscatter < -15 dB (water appears dark in radar)
sar_water = s1.lt(-15).rename('SAR_Water')

# Inundation extent: SAR-detected water MINUS permanent optical water
flood_extent = sar_water.And(optical_water.Not()).rename('Inundation_Only')

print("Optical Water Bands:", optical_water.bandNames().getInfo())
print("SAR Water Bands:", sar_water.bandNames().getInfo())
print("Flood segmentation complete.")

## 5. Calculate Real-World Flood Metrics

In [ ]:
# Sum flooded pixel areas within the AOI buffer region
pixel_area = ee.Image.pixelArea()
flood_area = flood_extent.multiply(pixel_area).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=region,
    scale=10,
    maxPixels=1e9
)

area_sq_km = ee.Number(flood_area.get('Inundation_Only')).divide(1e6).round()
print(f"Calculated Flood Extent in Kaziranga: {area_sq_km.getInfo()} sq. km")

## 6. Render Multi-Layer Geospatial Overlay

> Interactive Folium map with four layers:
>
> | Layer | Source | Color |
> |-------|--------|-------|
> | True Color Composite | Sentinel-2 B4/B3/B2 | Natural |
> | Permanent Water Bodies | NDWI (Optical) | 🔵 Blue |
> | SAR Water Detection | VV Backscatter | 🔷 Cyan |
> | Flood Inundation Extent | SAR − Optical | 🔴 Red |

In [ ]:
# Helper: Add Earth Engine tile layer to Folium map
def add_ee_layer(self, ee_image_object, vis_params, name):
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data &copy; Google Earth Engine',
        name=name,
        overlay=True,
        control=True
    ).add_to(self)

folium.Map.add_ee_layer = add_ee_layer

# Create map centered on Kaziranga
flood_map = folium.Map(location=[26.58, 93.17], zoom_start=11)

# Add visualization layers
flood_map.add_ee_layer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 True Color')
flood_map.add_ee_layer(optical_water.selfMask(), {'palette': ['0000FF']}, 'Optical Water (NDWI)')
flood_map.add_ee_layer(sar_water.selfMask(), {'palette': ['00FFFF']}, 'SAR Water (VV < -15dB)')
flood_map.add_ee_layer(flood_extent.selfMask(), {'palette': ['FF0000']}, 'Flood Inundation Extent')

flood_map.add_child(folium.LayerControl())
flood_map